![Platform+](images/banner.png)

# LAB 4: CREDITS & COST AWARENESS

_Hands-on Snowflake for data practitioners_

> **Persona track: Platform Admin** — same lab, your angle.

👉 **In this lesson, we'll understand, monitor, and estimate your Snowflake compute costs.**

📚 **By the end you'll be able to:**
- Identify what consumes Snowflake credits
- Read warehouse and query usage from ACCOUNT_USAGE
- Estimate cost before scaling a workload
- Right-size a warehouse and set auto-suspend

**Why this matters for admins:** You own the account's total credit spend. Understanding where credits go — and having the tools to cap, alert, and attribute them — is the difference between a governed environment and a surprise bill.

**Time:** ~35 min · **Lab interface:** Cortex Code (CoCo) is your tutor — ask it to explain or fix anything.
*Source data lives in the shared, read-only `PLATPLUS_DB`; everything you create lives in your own database, warehouse, and role — each named after your username (e.g. `USER_DB` / `USER_WH` / `USER_RL`).*

---
👉 **Connecting to your compute service:**
1. In the top bar, if no service is connected, click to connect and select your service.
2. Set your **role and warehouse** using the picker at the top-right — or just run the setup cell below, which sets them for you in SQL. (SQL cells run on the warehouse.)
3. Once the status shows **Connected** (green), run a cell with **Shift+Enter** (run + advance) or **Cmd/Ctrl+Enter** (run this cell only); the **▶** button works too.

> 📌 The service is a container that runs the notebook kernel and your **Python** cells. **SQL cells (and Snowpark pushdown) run on the query warehouse**, not the container. It stays running until it hits its idle timeout or you suspend it manually.
> 💡 **Best practice: Start a fresh chat for this lab.** Before you begin, open a fresh CoCo chat — click **New chat** (the **+** at the top of the panel). Each lab creates its own objects, so a fresh chat ensures CoCo isn't referencing tables or schemas from a prior lab that no longer exist. Shorter conversations also produce sharper answers and use fewer tokens.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('\"')
my_db = user + '_DB'
my_wh = user + '_WH'
my_role = user + '_RL'
my_analyst_role = user + '_ANALYST_RL'
print('Context ready. Signed in as ' + user)
print('Your workspace -> db: ' + my_db + ' | wh: ' + my_wh + ' | role: ' + my_role)

## Set Up Your Workspace
First, set up your **personal workspace**. The cell below switches to your own role, creates and uses your own warehouse, and creates your own database for what you'll build in this lab. Every object is named after your username, so everyone in the account stays out of each other's way.

In [ ]:
-- Set your personal context: your role + your own warehouse
USE ROLE {{my_role}};
CREATE WAREHOUSE IF NOT EXISTS {{my_wh}}
  WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE {{my_wh}};
-- Your personal database + the schema(s) this lab writes to
CREATE DATABASE IF NOT EXISTS {{my_db}};
CREATE SCHEMA IF NOT EXISTS {{my_db}}.SANDBOX;
-- Read from the shared, read-only course data
USE DATABASE PLATPLUS_DB;
USE SCHEMA ROASTERY_SALES;

## 1 · What Costs Money?
Snowflake bills primarily on two axes: **compute** (credits consumed by warehouses and serverless features, billed by the second) and **storage** (per TB/month, usually a much smaller line item). There are additional billing dimensions (cloud services, data transfer) covered in the [full pricing documentation](https://docs.snowflake.com/en/user-guide/credits) — for this lab, we focus on the two that make up 90%+ of most bills.

The table below shows what draws on the compute meter.

> 💡 **What's a credit?** A Snowflake **credit** is the unit of compute consumption — the currency you spend while warehouses run and serverless features work. Your account draws down credits from a prepaid contract or an on-demand balance, and their dollar value depends on your edition, cloud, and region. Everything in this lab is ultimately measured in credits — for reference, an X-Small warehouse burns **1 credit per hour**.

| Service | When it burns credits |
|---|---|
| Virtual warehouse | Every second it runs (60-second minimum each time it resumes) |
| Snowpipe (serverless) | Per GB ingested (fixed credits/GB) |
| Scheduled tasks (serverless) | Per execution |
| Cortex AI functions | Per token / API call |
| Storage | Per TB/month (on daily-average storage) — a separate, lower meter |

> 📌 **Your #1 cost lever:** a warehouse costs credits *only while it runs* — an idle, auto-suspended warehouse costs **zero**. Setting auto-suspend well is the simplest, highest-leverage control you have.

## 2 · Monitor Your Costs (Part 1)
Snowflake's [`SNOWFLAKE.ACCOUNT_USAGE`](https://docs.snowflake.com/en/sql-reference/account-usage) schema is a built-in telemetry database of everything that has happened in your account. Three telemetry areas cover most cost investigation work.

> 🔐 **Access note.** `ACCOUNT_USAGE` is restricted by default — a data analyst normally wouldn't have it, but a platform admin would. For this lab your role has been granted two read-only Snowflake database roles: **`USAGE_VIEWER`** (warehouse & credit usage) and **`GOVERNANCE_VIEWER`** (query history). Snowflake recommends these fine-grained database roles over the older, broad `IMPORTED PRIVILEGES` grant precisely so you don't hand out more access than you intend. In production, this access is typically an admin or governance-team privilege.

> ⏱️ **Expect a lag.** `ACCOUNT_USAGE` views are not real-time — they refresh on a delay that varies by view ([see current latencies](https://docs.snowflake.com/en/sql-reference/account-usage#account-usage-views)): `WAREHOUSE_METERING_HISTORY` up to ~3 hours, `QUERY_HISTORY` up to ~45 minutes, `QUERY_ATTRIBUTION_HISTORY` up to ~8 hours, and the Cortex usage views up to ~1 hour. An empty or incomplete result in a fresh training account is normal.

### 2a · Warehouse-level spend
[`WAREHOUSE_METERING_HISTORY`](https://docs.snowflake.com/en/sql-reference/account-usage/warehouse-metering-history) records credits consumed per warehouse, aggregated by hour. The query below rolls the last 7 days up per warehouse.

In [ ]:
SELECT warehouse_name,
       ROUND(SUM(credits_used), 3) AS credits_7d
FROM SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY
WHERE start_time >= DATEADD(day, -7, CURRENT_TIMESTAMP())
GROUP BY 1
ORDER BY credits_7d DESC;

> 🔎 **Review the output.** Which warehouse used the most credits? An empty result just means the account is quiet — normal in a fresh training environment.

### 2b · Per-query cost

**Why this matters for admins:** Per-query attribution lets you trace a cost spike to the exact user and query — the evidence you need for chargeback or for a conversation about query optimization.
[`QUERY_ATTRIBUTION_HISTORY`](https://docs.snowflake.com/en/sql-reference/account-usage/query_attribution_history) records a credit cost for each individual query. It does **not** include query text — for that you join to `QUERY_HISTORY`. Together they let you find the most expensive pipeline steps or ad-hoc queries.

In [ ]:
-- A query can appear as several rows in QUERY_ATTRIBUTION_HISTORY (e.g. if it
-- spans a warehouse resize), so we SUM credits per query_id first, then join
-- to QUERY_HISTORY for the text and execution details.
WITH qa AS (
    SELECT query_id,
           MAX(warehouse_name)              AS warehouse_name,
           SUM(credits_attributed_compute)  AS credits_attributed_compute
    FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_ATTRIBUTION_HISTORY
    WHERE start_time >= DATEADD(day, -7, CURRENT_TIMESTAMP())
    GROUP BY query_id
)
SELECT LEFT(qh.query_text, 70)                  AS query_preview,
       qa.warehouse_name,
       ROUND(qa.credits_attributed_compute, 5)  AS credits,
       ROUND(qh.execution_time / 1000.0, 1)      AS secs,
       ROUND(qh.bytes_scanned / 1024 / 1024, 1)  AS mb_scanned,
       qh.query_type
FROM qa
JOIN SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY qh
  ON qa.query_id = qh.query_id
ORDER BY credits DESC
LIMIT 10;

> 🔎 **Review the output.** Credits at the query level are small numbers — a few thousandths is typical for an XSMALL warehouse. But across hundreds of users and thousands of daily queries, they aggregate into the warehouse-level spend you saw in Section 2a. The value here is **relative comparison** — which query patterns are disproportionately expensive, and who's running them?

### 2c · AI and Cortex service costs

**Why this matters for admins:** AI costs are new and growing fast. Tracking them separately from warehouse costs lets you set distinct budgets and catch runaway AI usage before it dominates the bill.
Snowflake provides dedicated ACCOUNT_USAGE views for each AI service. Here are the current-generation views to use (older views like `CORTEX_FUNCTIONS_QUERY_USAGE_HISTORY` are deprecated):

| View | What it tracks |
|---|---|
| [`CORTEX_AI_FUNCTIONS_USAGE_HISTORY`](https://docs.snowflake.com/en/sql-reference/account-usage/cortex_ai_functions_usage_history) | All Cortex AI SQL functions (AI_COMPLETE, AI_CLASSIFY, AI_SENTIMENT, etc.) — per query, function, and model |
| [`CORTEX_CODE_SNOWSIGHT_USAGE_HISTORY`](https://docs.snowflake.com/en/sql-reference/account-usage/cortex_code_snowsight_usage_history) | Cortex Code (CoCo) usage inside Snowsight |
| [`CORTEX_ANALYST_USAGE_HISTORY`](https://docs.snowflake.com/en/sql-reference/account-usage/cortex_analyst_usage_history) | Cortex Analyst (natural language to SQL) |
| [`CORTEX_AGENT_USAGE_HISTORY`](https://docs.snowflake.com/en/sql-reference/account-usage/cortex_agent_usage_history) | Cortex Agents API |

These usage views refresh with a latency of up to ~1 hour (see the access & latency note in Section 2). **We'll explore Cortex AI functions themselves in Lab 5** — here we're only looking at their *cost*. 👉 **Try the example below:**

In [ ]:
-- AI function credit consumption by function and model (last 7 days)
SELECT function_name,
       model_name,
       ROUND(SUM(credits), 4)         AS credits_7d,
       COUNT(DISTINCT query_id)        AS queries
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY
WHERE start_time >= DATEADD(day, -7, CURRENT_TIMESTAMP())
GROUP BY 1, 2
ORDER BY credits_7d DESC;

> 🔎 **Review the output.** An empty result means no Cortex AI functions ran in this period — normal in a fresh training account. In production, this query tells you exactly which functions and models are driving AI credit spend, down to the individual query if needed. Different models have vastly different per-token costs — see the [Cortex AI Functions cost reference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-cost) for current rates by model and function.

**CoCo (Cortex Code) usage in Snowsight.** The query below sums CoCo usage **over the last 7 days**, one row per user. It reads the source columns `TOKEN_CREDITS` (credit cost), `TOKENS` (volume) and `USAGE_TIME`, and returns them under friendlier aliases you'll see in the result: **`CREDITS_7D`**, **`TOKENS_7D`** and **`REQUESTS`** (a count of CoCo calls).

What's a **token**? Language models don't read whole words — they break text into *tokens*, chunks of roughly ¾ of a word (about 4 characters of English). Both your prompt and the model's response are measured in tokens, and token volume is what drives AI credit cost. `TOKENS_7D` is the total tokens processed; `CREDITS_7D` converts that volume into credits.

In [ ]:
-- CoCo usage inside Snowsight (chat, SQL generation, code assist)
SELECT user_name,
       ROUND(SUM(token_credits), 4) AS credits_7d,
       SUM(tokens)                  AS tokens_7d,
       COUNT(*)                     AS requests
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_CODE_SNOWSIGHT_USAGE_HISTORY
WHERE usage_time >= DATEADD(day, -7, CURRENT_TIMESTAMP())
GROUP BY 1
ORDER BY credits_7d DESC;

> 🔎 **Review the output.** An empty result means no CoCo Snowsight sessions ran in this 7-day period. **`CREDITS_7D`** is the credit cost, **`TOKENS_7D`** the token volume, and **`REQUESTS`** the number of CoCo calls. For context: a typical CoCo conversation (a few prompts + responses) uses roughly 2,000–5,000 tokens — so if you see `TOKENS_7D` in the hundreds of thousands, that represents many interactions across the period.

### 2d · Snowsight Cost Management
Snowsight has a built-in **Cost Management** page that shows the same data without SQL. Navigate to **Admin → Cost Management** and use the controls:

- **Consumption tab, Usage Type = Compute**: warehouse credit trend by day — the same data as `WAREHOUSE_METERING_HISTORY`, visualised
- **Consumption tab → set the *View by* control (top-right of the chart) to *By Service*:** consumption splits into warehouse usage plus each serverless feature (e.g. `WAREHOUSE_METERING`, `PIPE`, and the serverless Cortex/AI features) as its own line — serverless spend visible without SQL
- **Account Overview tab**: summary cards for total spend, budget utilisation, anomaly alerts

> 📌 **What you cannot see in the UI:** per-query credit attribution is not surfaced in Snowsight. `QUERY_ATTRIBUTION_HISTORY` (Section 2b) is the only way to get cost at the individual query level.

## 3 · Estimate Before You Build (Part 2)
Monitoring tells you what *has* happened. Estimation helps you plan *before* you build. Snowflake recommends selecting a starting warehouse size based on **workload type and query complexity**, not on row count alone — then adjusting based on observed performance signals.

### How Snowflake recommends selecting warehouse size

From the [Warehouse considerations](https://docs.snowflake.com/en/user-guide/warehouses-considerations) docs: **start with the size that matches your workload type, then adjust based on observed performance.** Per-second billing means you can upsize, run a job, and suspend without committing permanently. And 'larger is not necessarily faster' for small, basic queries.

**Starting size by workload type:**

| Workload | Starting size | Key sizing factor |
|---|---|---|
| Development, testing, PoC | X-Small | Sufficient for exploration and Snowsight UI queries |
| Interactive analytics (small datasets) | X-Small – Small | Single-table queries, light filters |
| Regular reporting (multi-table JOINs, GBs scanned) | Small – Medium | Query complexity and bytes scanned |
| Heavy ELT, large transforms | Medium – Large | Data volume in bytes, not row count |
| Very large-scale production workloads | Large – X-Large | Experiment across sizes and compare runtime |

**Credit rates.** The table below shows the published **Gen1** per-hour rates (the in-docs baseline). **Gen2 standard warehouses are now the default for new accounts in most AWS/Azure regions** — they run on faster hardware at a **higher credit rate: roughly ~35% more on AWS/GCP and ~25% more on Azure** than the Gen1 figures below. Existing accounts may still be on Gen1 unless explicitly upgraded. The doubling-per-size-step pattern still holds within Gen2's own rates. Because Gen2 finishes many queries faster it consumes fewer seconds, so whether it costs more *in practice* depends on the workload — Snowflake's guidance is to **test, don't assume faster is cheaper**. For exact current numbers see the [Gen2 warehouses documentation](https://docs.snowflake.com/en/user-guide/warehouses-gen2), which links the Snowflake Service Consumption Table.

| Size | Credits/hr (Gen1 ref) | Credits/sec |
|---|---|---|
| X-Small | 1 | 0.0003 |
| Small | 2 | 0.0006 |
| Medium | 4 | 0.0011 |
| Large | 8 | 0.0022 |
| X-Large | 16 | 0.0044 |

### Adaptive Warehouses — sizing without the guesswork
**Adaptive Warehouses** are a newer warehouse type (Enterprise Edition, select AWS regions) that remove the sizing decision entirely. Instead of you picking XS/S/M and hand-tuning auto-suspend, Snowflake **allocates compute per query automatically** and scales resources up or down to match each workload as it runs.

- **Different billing model:** you're charged per query based on the compute actually consumed — not per-second for a fixed-size warehouse that may sit partly idle. (Per-query credit usage for Adaptive Warehouses shows up in the `QUERY_METERING_HISTORY` view.)
- **Good fit:** varied, unpredictable analytical workloads and mixed ELT + BI, where right-sizing by hand is a constant chore.
- **Not for:** HTAP (hybrid table) workloads or Snowpark-optimised (ML / memory-heavy) workloads.

> 📌 If Adaptive Warehouses are available in your region, evaluate them before manually sizing — they can remove the size-tuning cycle for mixed analytical workloads. The manual-sizing guidance in this lab still applies to standard (Gen1/Gen2) warehouses, which remain the default elsewhere.

**Step 1 — see how much data this workload touches.** Now that you know how to read cost telemetry and what signals to watch for, you're ready to estimate costs for a workload *before* you run it. The first input is data volume: how many rows does your workload scan? Run the query below to find out for the Roastery orders table.

> 📌 Table size in bytes and query complexity matter more than row count when choosing a warehouse size — but row count gives you a useful order-of-magnitude anchor.

In [ ]:
-- Data volume context for this workload
SELECT COUNT(*) AS row_count
FROM PLATPLUS_DB.ROASTERY_SALES.ORDERS;

> 🔎 **Review the output.** The Roastery dataset is small. For the estimation exercise, use the workload-type table above — pick the tier that matches how you'd use this data (exploration vs. reporting vs. heavy ELT), not a tier derived from this row count.

👉 **Step 2 — estimate monthly credits.** Pick the workload tier from the table above, set your expected active hours, and run the cell.

In [ ]:
# YOUR TURN: pick your starting workload tier and working hours, then run
# Step 1 — choose a tier from the workload-type table above
workload_tier        = 1   # <- 1=X-Small  2=Small  3=Medium  4=Large  5=X-Large

# Step 2 — estimate your working hours per day
hours_active_per_day = 8   # <- how many hours per day the warehouse is running

# -- official Gen1 credit rates (don't change below this line) ----------
WH_SIZES = {1: ('X-Small', 1), 2: ('Small', 2), 3: ('Medium', 4), 4: ('Large', 8), 5: ('X-Large', 16)}
wh_size, cph = WH_SIZES.get(workload_tier, ('X-Large', 16))
monthly_credits = cph * hours_active_per_day * 30

print(f'Starting WH size:      {wh_size}')
print(f'Credits/hr:            {cph}')
print(f'Est. monthly credits:  {monthly_credits}  ({hours_active_per_day}h/day x 30 days)')
print()
print('This is a starting estimate. Run your actual workload, check for spilling in')
print('QUERY_HISTORY.bytes_spilled_to_local_storage, and resize based on those signals.')

> 🔎 **Review the output.** Try changing the tier or hours: what does a Medium warehouse running 12h/day cost vs an X-Small running 8h/day?

👉 **Step 3 — convert to dollars.** Your estimate shows monthly credits. To see the dollar cost for your cloud provider and region:

[**Snowflake Pricing Calculator →**](https://www.snowflake.com/en/pricing-options/calculator)

Enter your cloud provider, region, and the monthly credits from the estimator above. This gives you a realistic cost range before you build.

> 📌 **Note your estimate.** Write down or screenshot your recommended WH size and monthly credits from above — this is your starting point for cost discussions and warehouse configuration decisions.

## 4 · Control Cost: Right-size + Auto-suspend

**Why this matters for admins:** These are the two settings you'll audit across every warehouse in the fleet. A single warehouse with AUTO_SUSPEND=0 (never suspends) can quietly burn credits indefinitely.
Two warehouse settings that are always worth checking:

- **`WAREHOUSE_SIZE`** — bigger warehouses cost more per hour but finish jobs faster. Match the size to the workload, not the other way around.
- **`AUTO_SUSPEND`** — idle timeout in seconds before the warehouse suspends itself. `60` seconds is a sensible default for interactive workloads.

**Why not to set auto-suspend too aggressively:** a suspended warehouse drops its local disk cache. The first few queries after resume run cold — slower until the cache rebuilds. If your workload has regular short gaps (e.g. a dashboard refreshing every 2 minutes), aggressive auto-suspend triggers repeated suspend-resume cycles, each billed a minimum 60 seconds. Match the suspend window to your actual inter-query gap pattern.

**Auto-resume:** a warehouse with `AUTO_RESUME = TRUE` (the default) restarts automatically the moment any query is issued against it — typically 1–2 seconds. Users see no error; they simply wait slightly longer for the first query after an idle period.

Set both on your own warehouse now.

In [ ]:
ALTER WAREHOUSE {{my_wh}} SET AUTO_SUSPEND = 60;
ALTER WAREHOUSE {{my_wh}} SET WAREHOUSE_SIZE = XSMALL;
SHOW WAREHOUSES LIKE '{{my_wh}}';

> 🔎 **Review the output.** AUTO_SUSPEND should now read 60. Small settings like this, applied consistently across a fleet, add up.

### Spot under-sizing: check for spilling

**After you've run your workload,** confirm your sizing choice with evidence. The clearest sign a warehouse is **too small** is **spilling**: when a query runs out of memory it writes intermediate results to disk (local first, then remote). Learn more: [What spilling is and why it hurts](https://community.snowflake.com/s/article/Performance-impact-from-local-and-remote-disk-spilling) · [Finding and fixing spilling queries](https://docs.snowflake.com/en/user-guide/performance-query-warehouse-memory). Spilling is one tool in your right-sizing arsenal, and you can see it two ways:

- **In the UI:** open a query's **Query Profile** and read the **Bytes spilled to local/remote storage** statistics.
- **Programmatically:** query the `bytes_spilled_to_*` columns in `SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY` (the cell below).

> 💡 **First line of defense: read the query, not just the size.** Before upsizing, check the query itself — pull only the columns and rows you need, and confirm every join has a condition. A poorly-written query (missing filter, accidental cross join) is a common root cause of spilling; a bigger warehouse just pays to move the same waste faster.

How to read the result:

- `spill_local_mb > 0`: mildly undersized — upsize one step
- `spill_remote_mb > 0`: severely undersized — upsize 2+ steps
- **No rows**: no queries spilled — the warehouse is *big enough* for current workloads (on its own this doesn't prove it isn't over-sized; use the metering + query-cost views to check for waste)

In [ ]:
-- Check for any spilling in the last 7 days (the signal to upsize)
SELECT LEFT(query_text, 150)              AS query,
       warehouse_name,
       ROUND(bytes_spilled_to_local_storage / 1024 / 1024, 2)    AS spill_local_mb,
       ROUND(bytes_spilled_to_remote_storage / 1024 / 1024, 2)   AS spill_remote_mb,
       query_acceleration_bytes_scanned AS qas_bytes,
       ROUND(execution_time / 1000.0, 1) AS secs
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE start_time >= DATEADD(day, -7, CURRENT_TIMESTAMP())
  AND (bytes_spilled_to_local_storage > 0
    OR bytes_spilled_to_remote_storage > 0)
ORDER BY bytes_spilled_to_local_storage DESC
LIMIT 10;

> 🔎 **Review the output.** No rows = nothing spilled, so the warehouse is big enough (not necessarily right-sized — an over-sized warehouse also shows no spill; cross-check with the metering/query-cost views). If you do see rows, read the two spill columns differently:
>
> | Column | Severity | What it means |
> |--------|----------|---------------|
> | `spill_local_mb` > 0 | ⚠️ Minor | Query overflowed memory to local SSD — slower but tolerable for occasional large queries |
> | `spill_remote_mb` > 0 | 🔴 Significant | Query overflowed to cloud storage — major performance hit, strong signal to upsize the warehouse |
>
> A non-zero **qas_bytes** means Query Acceleration Service helped that query — spill on a QAS-accelerated query is still a real undersizing signal, but note QAS was already compensating for it.
>
> **Rule of thumb:** Occasional local spill on your largest query is acceptable. Any remote spill, or frequent local spill, means the warehouse is undersized for that workload — step the size up and re-check.

### Try a CoCo skill: `/warehouse`

Skills were introduced in Lab 2. Here's the first skill use in this lab. `/warehouse` has structured knowledge of Gen2 architecture, credit billing mechanics, sizing trade-offs, and auto-suspend patterns.

👉 Open the **CoCo panel**, type `/` to open the skill picker, select **warehouse**.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">/warehouse For a platform administrator running 10 queries per day against the Roastery orders table (small dataset, XSMALL warehouse with AUTO_SUSPEND=60), is this the right configuration? What would you recommend for a mixed workload of ad-hoc queries and scheduled daily reports?
</pre>

> 📌 **What to expect:** The skill validates the configuration, explains Gen2 billing mechanics, and advises on the auto-suspend vs cache warmth trade-off.

## 5 · CoCo Cost Intelligence — Ask the AI
CoCo has a built-in **cost-intelligence** skill that runs ACCOUNT_USAGE queries for you and adds a plain-English summary — the same data as Sections 2a–2c, surfaced through AI.

**How to invoke the skill:**
1. Open the **CoCo panel** (chat icon, top-right in Snowsight)
2. Type `/` — a skill picker appears
3. Select **cost-intelligence**
4. Ask your questions

👉 **Try these three questions in order:**

**Q1 — Biggest spenders:**

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">/cost-intelligence What are my top 3 warehouses by credit spend in the last 30 days?
</pre>

> 📌 After CoCo answers, find the SQL it generated (you may need to unfurl its execution steps), click **Copy**, paste it into the cell below, amend the date range to show the last **90 days**, and add `LIMIT 10`. Run it. Is your warehouse in this list?

In [ ]:
-- YOUR TURN: paste CoCo's SQL on the blank line below, then:
-- 1. Change the date range to 90 days (e.g. DATEADD(day, -90, CURRENT_TIMESTAMP()))
-- 2. Add LIMIT 10 at the end to show the top 10 results


**Q2 — Drill into a warehouse:**

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">/cost-intelligence How many credits per day is {{my_wh}} consuming and what can I do to reduce it?
</pre>

**Q3 — AI and service costs:**

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">/cost-intelligence Show me credit spend by service type in the last 30 days, including any Cortex AI or CoCo usage.
</pre>

> 🔎 **Review CoCo's output.** Notice it generates SQL, executes it, and adds a summary. The skill is doing exactly what you did manually in Sections 2a–2c — this is how AI assistance accelerates the cost-monitoring workflow.

## 6 · Your Angle — Platform Admin
As a platform admin you own the *fleet's* cost. This angle walks the full admin loop: **see** where credits go and **allocate** them to teams, **guard** spend at every level (query, warehouse, account), then **diagnose** the expensive workloads. Start by allocating your fleet's spend to cost centers.

**See your fleet spend — allocated by cost center:**
Section 2 showed credits per *warehouse*. As an admin you also have to answer *"which team or business unit does this spend belong to?"* — that's **chargeback**. The standard pattern: put an **object tag** (here, `COST_CENTER`) on each warehouse, then join `WAREHOUSE_METERING_HISTORY` to `TAG_REFERENCES` to roll spend up by tag. This account has a few warehouses pre-tagged for the demo.

In [ ]:
-- Fleet spend rolled up by COST_CENTER object tag (last 30 days)
SELECT COALESCE(tr.tag_value, '(unallocated)')   AS cost_center,
       ROUND(SUM(wmh.credits_used), 3)           AS credits_30d,
       COUNT(DISTINCT wmh.warehouse_name)         AS warehouses
FROM SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY wmh
LEFT JOIN SNOWFLAKE.ACCOUNT_USAGE.TAG_REFERENCES tr
       ON tr.object_name = wmh.warehouse_name
      AND tr.domain = 'WAREHOUSE'
      AND tr.tag_name = 'COST_CENTER'
WHERE wmh.start_time >= DATEADD(day, -30, CURRENT_TIMESTAMP())
GROUP BY 1
ORDER BY credits_30d DESC;

> 🔎 **Review the output.** Tagged warehouses roll up under their cost center; everything else lands in **`(unallocated)`**. In a real FinOps practice the goal is to shrink `(unallocated)` toward zero so every credit is attributable to a team. The three `COST_DEMO_*` warehouses are exercised **once at account setup**, so their credits are small and may be a couple of weeks old — this query looks back **30 days** to catch that seed activity. (ACCOUNT_USAGE latency also applies — metering can take a couple of hours to appear after setup.) If the account was provisioned more than ~30 days ago you may still see **only `(unallocated)`**; the *pattern* (tag → roll-up) is the point, not the specific numbers.

### Cost guardrail 1: Resource Monitors
A **[resource monitor](https://docs.snowflake.com/en/user-guide/resource-monitors)** tracks the credits a warehouse (or the whole account) consumes over a set window and can **notify** you — or **suspend** the warehouse — when usage crosses a threshold you define. It's your reactive cost ceiling: the "don't let a runaway warehouse blow the budget" guardrail.

Resource monitors are **account-level objects** — only `ACCOUNTADMIN` can create one, attach it to a warehouse, or even see it. So this lab drives those steps through **owner's-rights stored procedures** that perform the admin action on your behalf and grant you visibility of the result.

**Step 1 — create your monitor.**
The proc below creates `<YOUR_USERNAME>_RM` with a **100-credit daily quota** (matching the account's standard per-warehouse monitors), a **notify at 75%** and a **suspend at 100%** — then grants your role `MONITOR` on it so you can see it. 100 credits/day is far above anything this lab spends, so it won't interrupt you.

> 💡 **Why a procedure?** Creating resource monitors requires ACCOUNTADMIN — a privilege you don't (and shouldn't) have. The stored procedure `CREATE_DEMO_RESOURCE_MONITOR` runs with **owner's rights** (ACCOUNTADMIN) on your behalf. Here's what it executes internally:
>
> ```sql
> -- What the procedure runs as ACCOUNTADMIN:
> CREATE RESOURCE MONITOR <user>_RM WITH CREDIT_QUOTA = 100 FREQUENCY = DAILY
>   TRIGGERS ON 75 PERCENT DO NOTIFY
>            ON 100 PERCENT DO SUSPEND;
> ALTER WAREHOUSE <user>_WH SET RESOURCE_MONITOR = <user>_RM;
> GRANT MONITOR ON RESOURCE MONITOR <user>_RM TO ROLE <user>_RL;
> ```
>
> This is the same privilege-delegation pattern you saw in Lab 3 (future grants) — a controlled way to let non-admin roles trigger admin-level operations within guardrails.

In [ ]:
CALL PLATPLUS_DB.PUBLIC.CREATE_DEMO_RESOURCE_MONITOR(CURRENT_USER());

> 🔎 **Review the output.** The proc created your monitor *and* granted your role visibility of it. Because resource monitors are ACCOUNTADMIN-owned, without that `MONITOR` grant a `SHOW` would return nothing — you'll confirm it works in the next step.

**Step 2 — see your monitor.**
`SHOW RESOURCE MONITORS` lists the monitors your current role is allowed to see. Thanks to the grant in Step 1, yours now appears.

In [ ]:
SHOW RESOURCE MONITORS;

> 🔎 **Review the output.** You should see `<YOUR_USERNAME>_RM` — `credit_quota` 100, `frequency` DAILY, `notify_at` 75%, `suspend_at` 100%. A monitor set to SUSPEND at 100% stops every warehouse assigned to it once the quota is hit.

**Step 3 — attach the monitor to your warehouse.**
A monitor only acts once it's **attached** to a warehouse. Attaching is also an `ACCOUNTADMIN` action, so it runs through a proc that attaches `<YOUR_USERNAME>_RM` to `<YOUR_USERNAME>_WH`.

In [ ]:
CALL PLATPLUS_DB.PUBLIC.ATTACH_DEMO_RESOURCE_MONITOR(CURRENT_USER());

**Step 4 — verify the attachment.** Look at the `resource_monitor` column for your warehouse.

In [ ]:
SHOW WAREHOUSES LIKE '{{my_wh}}';

> 🔎 **Review the output.** The `resource_monitor` column now shows `<YOUR_USERNAME>_RM` — your warehouse is guarded. As it runs, the monitor's `used_credits` climbs toward the 100-credit daily quota, and at 100% it would auto-suspend. In production this attach is a one-liner — `ALTER WAREHOUSE <wh> SET RESOURCE_MONITOR = <name>` — but it's an `ACCOUNTADMIN` action, which is why the lab wraps it in a proc. The next cell detaches it so your warehouse runs freely in later labs.

In [ ]:
-- Cleanup: detach the monitor so your warehouse runs freely in other labs
CALL PLATPLUS_DB.PUBLIC.DETACH_DEMO_RESOURCE_MONITOR(CURRENT_USER());

### The other guardrail: Budgets
A Resource Monitor is a **reactive kill-switch** — it *suspends* warehouses when a credit quota is hit. A **Budget** is its **proactive** counterpart: it tracks spend against a target and **notifies** you (by email) as you approach it — without stopping anything — and it covers **serverless and storage**, not just warehouse credits. Mature accounts run both: Budgets for early warning, Resource Monitors as the hard stop.

| | Resource Monitor | Budget |
|---|---|---|
| Action when hit | **Suspends** warehouses (or notifies) | **Notifies** only — never suspends |
| Covers | Warehouse credits | Warehouses **+ serverless + storage** |
| Scope | Account or per-warehouse | Account, or a custom group of objects |
| Best for | Hard ceiling / enforcement | Early warning / FinOps tracking |
| Created by | ACCOUNTADMIN | ACCOUNTADMIN (or a granted role) |

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">/cost-intelligence Explain how I'd set up a budget to alert me before my account passes 100 credits this month, and how that differs from the resource monitor I just created.
</pre>

### A query-level guardrail: statement timeout
Resource Monitors and Budgets act at the account / warehouse level. The finest-grained guardrail is **`STATEMENT_TIMEOUT_IN_SECONDS`** — it auto-cancels any single query that runs longer than the limit, capping the damage from a runaway query. Unlike a monitor, you can set it on a warehouse **you own**, so you can run this one yourself. (It can also be set at the account or session level.)

We'll set a deliberately short **20-second** limit so you can watch it fire — in production you'd pick something higher (often minutes).

In [ ]:
-- Short 20s cap on your warehouse (low on purpose so the demo below triggers fast)
ALTER WAREHOUSE {{my_wh}} SET STATEMENT_TIMEOUT_IN_SECONDS = 20;
SHOW PARAMETERS LIKE 'STATEMENT_TIMEOUT_IN_SECONDS' IN WAREHOUSE {{my_wh}};

👉 **Watch it fire.** The next query does nothing but sleep for 30 seconds — longer than your 20s cap — so Snowflake auto-cancels it. It will run for ~20s and then error; that's the guardrail working, not a mistake.

In [ ]:
-- Intentionally runs past the 20s cap -> Snowflake auto-cancels it
SELECT SYSTEM$WAIT(30);

That query was killed at 20 seconds with a *"statement reached its timeout"* error — exactly what you want a runaway query to do. Now reset the cap so it doesn't interfere with later labs.

> ⚠️ **Why didn't it cancel at exactly 20 seconds?** Snowflake checks timeout conditions periodically, not continuously — so the actual cancellation may occur a few seconds after the threshold. You might see 22s, 26s, or even 30s before the kill fires. This is normal behavior, not a bug. The guardrail still works — it just has a few seconds of enforcement lag.

In [ ]:
-- Reset: remove the demo cap so your warehouse behaves normally in other labs
ALTER WAREHOUSE {{my_wh}} UNSET STATEMENT_TIMEOUT_IN_SECONDS;

> 🔎 **Guardrails at three levels.** You've now capped cost at every level — **query** (statement timeout), **warehouse / account** (resource monitor), and **account-wide spend** (budget). Together they stop a runaway query, a runaway warehouse, and a runaway month.

**Snowsight Cost Management (fleet view):**
Open **Admin → Cost Management** in Snowsight for a visual fleet view:
- **Account Overview** tab: total spend, trends, top spenders, and anomalies at a glance
- **Consumption** tab: drill into per-warehouse or per-service usage (set *View by* to *Resource* or *By Service*)
- **Resource Monitors** tab: quota vs. actual spend across all monitors

This is the view your finance team or FinOps partner will reference for chargebacks.

### Let CoCo do the cost work: two admin skills
The rest of this section uses two CoCo skills built for exactly this job. Invoke each by typing `/` in the CoCo panel and selecting it:
- **`/cost-intelligence`** — pulls ACCOUNT_USAGE cost data for you and summarizes it in plain English (spend by warehouse or service, top spenders, reduction ideas). Your *"where are the credits going?"* skill.
- **`/workload-performance-analysis`** — diagnoses *why* a warehouse is expensive (query patterns, spilling, pruning) and recommends fixes. Your *"why is it costing that?"* skill.

The admin workflow: use `/cost-intelligence` to find the spend, then `/workload-performance-analysis` to explain and fix it.

**First — `/cost-intelligence` for a fleet cost report:**

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">/cost-intelligence Generate a 30-day cost report showing credit spend per warehouse for this account.
</pre>

### Performance diagnosis with CoCo.
Cost numbers tell you *what* is being spent. Performance analysis tells you *why*. As admin, `/workload-performance-analysis` is the link between the two.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">/workload-performance-analysis analyze {{my_wh}} — identify the worst-performing query patterns, explain the cost drivers, and give me your top recommendations for reducing credit consumption
</pre>

> 📌 **What to expect:** WPA gives you a performance health summary for the warehouse — top bottlenecks (spilling, poor pruning, QAS candidates) and warehouse-level recommendations. This is the diagnostic layer behind the numbers from `WAREHOUSE_METERING_HISTORY`.

> 🔎 **Review the output.** WPA output + WAREHOUSE_METERING_HISTORY + Resource Monitor together form the complete admin cost governance workflow: measure spend → diagnose root cause → set guardrails.

## Check Your Work
Run the next cell to **check your work**. It queries your own database to confirm you created the objects and rows this lab asked for, and prints a ✅ / ⛔ result for each — no need to read the code.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
mon_name = session.get_current_user().strip('"') + '_RM'
# Verify via SHOW RESOURCE MONITORS (works through the MONITOR USAGE privilege granted in this lab).
# This deliberately avoids SNOWFLAKE.ACCOUNT_USAGE.RESOURCE_MONITORS, which has up-to-2h latency —
# a monitor you just created would not appear there yet and would cause a false ⛔.
try:
    n = len(session.sql(f"SHOW RESOURCE MONITORS LIKE '{mon_name}'").collect())
    ok = n >= 1
except Exception:
    n, ok = None, False
icon = '✅' if ok else '⛔'
detail = f'(found {n})' if n is not None else '(not found)'
print(f'{icon}  Your resource monitor {mon_name} exists  {detail}')
print('-' * 52)
if ok:
    print('🎉 CHECK PASSED — lab complete! Nice work.')
else:
    print('⚠️  Not found yet — re-run the CREATE_DEMO_RESOURCE_MONITOR cell above, then run this cell again.')

### Knowledge check 🧠
Check your understanding with interactive quiz questions below. Each quiz cell has a widget that presents a multiple‑choice question about Snowflake functionality.

**Instructions:**
1. Hover your cursor over the notebook cell to reveal additional controls.
1. Click the ▶️ **Play button** on the right side of each quiz cell to run it.
1. Select your answer from the options provided.
1. Review the feedback before moving on.

In [ ]:
from resources.knowledgecheck import quiz
quiz.display_question("q1")

In [ ]:
from resources.knowledgecheck import quiz
quiz.display_question("q2")

In [ ]:
from resources.knowledgecheck import quiz
quiz.display_question("q3")

In [ ]:
from resources.knowledgecheck import quiz
quiz.display_question("q4")

In [ ]:
from resources.knowledgecheck import quiz
quiz.display_question("q5")

In [ ]:
from resources.knowledgecheck import quiz
quiz.display_question("q6")

### Key takeaways. ❄️
- Compute burns credits; a *suspended* warehouse costs nothing (a running-but-idle one still bills) — auto-suspend is your simplest lever
- WAREHOUSE_METERING_HISTORY and QUERY_ATTRIBUTION_HISTORY show where credits go
- Match warehouse size to data volume and workload type before scaling
- Estimate monthly credits before building; use the Snowflake Pricing Calculator to convert to dollars

### Notebook completion. 🏁
You're done — your notebook is saved automatically.

> 💡 **Moving on to another lab?** Shut down this notebook's **kernel** — click **Connected** at the top → **Shut down kernel**. All your lab notebooks share one compute service, so clearing a finished kernel frees up memory for the next lab. Your service stays running, ready in seconds.

> 💡 **Finished for the day?** **Suspend your service** — click **Connected** → hover your service name → **Suspend**. This stops compute; it resumes automatically next time you run a cell.